# Tinygrad: Entendiendo el Autograd y Grafos de Computacion desde Cero
## Bloque B - Ingenieria y Frameworks (Nivel Tecnico/Codigo)
**Universidad Privada del Norte - Escuela de Posgrado**

---

### Objetivo
Construir paso a paso un motor de diferenciacion automatica (autograd) desde cero, entender como funcionan los grafos de computacion, y analizar como tinygrad implementa estos conceptos para ejecutar modelos reales como LLaMA y Stable Diffusion.

### Referencia
- Repo: https://github.com/tinygrad/tinygrad
- Inspirado en micrograd de Andrej Karpathy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from graphviz import Digraph
import warnings
warnings.filterwarnings('ignore')

print('Entorno listo')

---
## Parte 1: Que es la Diferenciacion Automatica?

La **diferenciacion automatica (autodiff)** es la tecnica fundamental que permite entrenar redes neuronales. No es diferenciacion simbolica (como Mathematica) ni diferenciacion numerica (como diferencias finitas). Es un metodo exacto y eficiente basado en la **regla de la cadena**.

### Modos de Autodiff:
- **Forward mode**: Propaga derivadas desde las entradas hacia las salidas
- **Reverse mode (backpropagation)**: Propaga gradientes desde las salidas hacia las entradas

Para redes neuronales usamos **reverse mode** porque tipicamente tenemos muchos parametros (entradas) y una sola salida (loss).

### La Regla de la Cadena
Si $z = f(g(x))$, entonces:
$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$$

donde $y = g(x)$.

---
## Parte 2: Construyendo un Motor Autograd desde Cero

Vamos a implementar una clase `Value` que:
1. Almacena un valor escalar
2. Registra las operaciones realizadas (grafo de computacion)
3. Calcula gradientes automaticamente via backpropagation

In [ ]:
class Value:
    """Motor de autograd escalar - inspirado en micrograd/tinygrad.
    Cada Value es un nodo en el grafo de computacion."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0  # Derivada de la salida respecto a este nodo
        self._backward = lambda: None  # Funcion de backprop
        self._prev = set(_children)  # Nodos padres en el grafo
        self._op = _op  # Operacion que creo este nodo
        self.label = label
    
    def __repr__(self):
        return f"Value({self.label}={self.data:.4f}, grad={self.grad:.4f})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # d(a+b)/da = 1, d(a+b)/db = 1
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            # d(a*b)/da = b, d(a*b)/db = a
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            # d(x^n)/dx = n * x^(n-1)
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __radd__(self, other):
        return self + other
    
    def __rmul__(self, other):
        return self * other
    
    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        
        def _backward():
            # d(relu(x))/dx = 1 si x > 0, 0 si x <= 0
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        t = np.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            # d(tanh(x))/dx = 1 - tanh(x)^2
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        e = np.exp(self.data)
        out = Value(e, (self,), 'exp')
        
        def _backward():
            # d(e^x)/dx = e^x
            self.grad += e * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        """Backpropagation usando orden topologico."""
        # Construir orden topologico
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # El gradiente de la salida respecto a si misma es 1
        self.grad = 1.0
        
        # Propagar gradientes en orden inverso
        for node in reversed(topo):
            node._backward()

print('Clase Value definida correctamente')
print('Operaciones soportadas: +, -, *, /, **, relu, tanh, exp')

### 2.1 Verificacion: Comparar con Derivadas Analiticas

In [ ]:
# Ejemplo 1: f(x,y) = x*y + x^2
# df/dx = y + 2x, df/dy = x
x = Value(3.0, label='x')
y = Value(4.0, label='y')

z = x * y + x ** 2
z.label = 'z'
z.backward()

print('f(x,y) = x*y + x^2')
print(f'x = {x.data}, y = {y.data}')
print(f'z = f(3,4) = 3*4 + 3^2 = {z.data}')
print(f'')
print(f'Gradientes calculados por autograd:')
print(f'  dz/dx = {x.grad:.4f} (esperado: y + 2x = 4 + 6 = 10.0)')
print(f'  dz/dy = {y.grad:.4f} (esperado: x = 3.0)')
print(f'')
assert abs(x.grad - 10.0) < 1e-6, 'Error en gradiente de x'
assert abs(y.grad - 3.0) < 1e-6, 'Error en gradiente de y'
print('VERIFICADO: Los gradientes coinciden con las derivadas analiticas')

In [ ]:
# Ejemplo 2: Verificacion con diferencias finitas (metodo numerico)
def verify_gradient(f, inputs, labels, h=1e-7):
    """Compara gradientes de autograd con diferencias finitas."""
    # Forward + backward con autograd
    vals = [Value(x, label=l) for x, l in zip(inputs, labels)]
    result = f(*vals)
    result.backward()
    autograd_grads = [v.grad for v in vals]
    
    # Diferencias finitas
    numerical_grads = []
    for i in range(len(inputs)):
        inputs_plus = list(inputs)
        inputs_plus[i] += h
        vals_plus = [Value(x) for x in inputs_plus]
        f_plus = f(*vals_plus).data
        
        inputs_minus = list(inputs)
        inputs_minus[i] -= h
        vals_minus = [Value(x) for x in inputs_minus]
        f_minus = f(*vals_minus).data
        
        numerical_grads.append((f_plus - f_minus) / (2 * h))
    
    print('Comparacion de gradientes:')
    print(f'{"Variable":<10} {"Autograd":>12} {"Numerico":>12} {"Error":>12}')
    print('-' * 48)
    for l, ag, ng in zip(labels, autograd_grads, numerical_grads):
        error = abs(ag - ng)
        status = 'OK' if error < 1e-4 else 'FALLO'
        print(f'{l:<10} {ag:>12.6f} {ng:>12.6f} {error:>12.2e} {status}')
    return all(abs(ag - ng) < 1e-4 for ag, ng in zip(autograd_grads, numerical_grads))

# Test con funcion compleja: f(a,b,c) = (a*b + c).tanh()
print('f(a,b,c) = tanh(a*b + c)')
result = verify_gradient(
    lambda a, b, c: (a * b + c).tanh(),
    [2.0, 3.0, -1.0],
    ['a', 'b', 'c']
)
print(f'\nTodas las verificaciones pasaron: {result}')

---
## Parte 3: Visualizacion del Grafo de Computacion

Cada operacion crea un nodo en un **DAG (Directed Acyclic Graph)**. Este grafo es la estructura que permite la backpropagation.

In [ ]:
def draw_graph(root, title='Grafo de Computacion'):
    """Visualiza el grafo de computacion de un Value."""
    dot = Digraph(format='png', graph_attr={'rankdir': 'LR', 'bgcolor': '#1a1a2e'})
    dot.attr('node', style='filled', fontcolor='white', fontsize='11')
    dot.attr('edge', color='#888888')
    
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    
    for n in nodes:
        label = f"{n.label}\ndata={n.data:.3f}\ngrad={n.grad:.3f}"
        if n._op == '':
            # Nodo hoja (input)
            dot.node(str(id(n)), label, shape='box', fillcolor='#0e4d92')
        else:
            # Nodo intermedio o salida
            color = '#2d6a4f' if n.grad != 0 else '#4a4e69'
            dot.node(str(id(n)), label, shape='ellipse', fillcolor=color)
            # Nodo de operacion
            op_id = str(id(n)) + '_op'
            dot.node(op_id, n._op, shape='circle', fillcolor='#e63946',
                    width='0.4', height='0.4', fontsize='12')
            dot.edge(op_id, str(id(n)))
            for child in n._prev:
                dot.edge(str(id(child)), op_id)
    
    return dot

# Crear un grafo de ejemplo: neurona simple
# z = w1*x1 + w2*x2 + b
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813, label='b')

x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'sum'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

o.backward()

print('Neurona: o = tanh(w1*x1 + w2*x2 + b)')
print(f'Salida: o = {o.data:.4f}')
print(f'\nGradientes:')
print(f'  do/dw1 = {w1.grad:.4f}')
print(f'  do/dw2 = {w2.grad:.4f}')
print(f'  do/dx1 = {x1.grad:.4f}')
print(f'  do/dx2 = {x2.grad:.4f}')

graph = draw_graph(o, 'Grafo: Neurona con tanh')
graph

---
## Parte 4: Red Neuronal Completa con Nuestro Autograd

Ahora construiremos una red neuronal funcional usando solo nuestra clase `Value`.

In [ ]:
import random
random.seed(42)

class Neuron:
    """Una neurona con pesos, bias y activacion."""
    def __init__(self, n_inputs, activation='relu'):
        self.w = [Value(random.uniform(-1, 1), label=f'w{i}') for i in range(n_inputs)]
        self.b = Value(random.uniform(-1, 1), label='b')
        self.activation = activation
    
    def __call__(self, x):
        # w . x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        if self.activation == 'relu':
            return act.relu()
        elif self.activation == 'tanh':
            return act.tanh()
        return act  # linear
    
    def parameters(self):
        return self.w + [self.b]


class Layer:
    """Capa de neuronas."""
    def __init__(self, n_inputs, n_outputs, activation='relu'):
        self.neurons = [Neuron(n_inputs, activation) for _ in range(n_outputs)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    """Multi-Layer Perceptron."""
    def __init__(self, n_inputs, layer_sizes):
        sizes = [n_inputs] + layer_sizes
        self.layers = []
        for i in range(len(layer_sizes)):
            act = 'relu' if i < len(layer_sizes) - 1 else 'linear'
            self.layers.append(Layer(sizes[i], sizes[i+1], act))
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# Crear red: 2 entradas -> 8 neuronas -> 8 neuronas -> 1 salida
model = MLP(2, [8, 8, 1])
n_params = len(model.parameters())
print(f'Red creada: MLP(2, [8, 8, 1])')
print(f'Total de parametros: {n_params}')
print(f'Arquitectura: 2 -> 8 (ReLU) -> 8 (ReLU) -> 1 (Linear)')

In [ ]:
# Dataset: Clasificacion de puntos en circulos concentricos
np.random.seed(42)
N = 100

# Clase 0: circulo interno
r1 = np.random.uniform(0, 1, N // 2)
theta1 = np.random.uniform(0, 2 * np.pi, N // 2)
X_inner = np.column_stack([r1 * np.cos(theta1), r1 * np.sin(theta1)])

# Clase 1: circulo externo
r2 = np.random.uniform(1.5, 2.5, N // 2)
theta2 = np.random.uniform(0, 2 * np.pi, N // 2)
X_outer = np.column_stack([r2 * np.cos(theta2), r2 * np.sin(theta2)])

X = np.vstack([X_inner, X_outer])
y = np.array([-1] * (N // 2) + [1] * (N // 2))  # -1 y +1

# Shuffle
idx = np.random.permutation(N)
X, y = X[idx], y[idx]

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(X[y == -1, 0], X[y == -1, 1], c='#e63946', s=30, label='Clase -1')
ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#457b9d', s=30, label='Clase +1')
ax.set_title('Dataset: Circulos Concentricos', fontsize=14, fontweight='bold')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Muestras: {N}, Features: 2, Clases: 2')

In [ ]:
# Entrenamiento con SGD manual usando nuestro autograd
losses = []
accuracies = []
lr = 0.05
epochs = 50

for epoch in range(epochs):
    # Forward pass: calcular predicciones
    preds = [model([Value(x[0]), Value(x[1])]) for x in X]
    
    # Hinge loss (SVM loss): max(0, 1 - y*pred)
    data_loss = sum((Value(1) + Value(-yi) * pred).relu() for yi, pred in zip(y, preds)) * Value(1.0 / N)
    
    # Regularizacion L2
    reg_loss = sum(p * p for p in model.parameters()) * Value(1e-4)
    total_loss = data_loss + reg_loss
    
    # Accuracy
    acc = sum(1 for yi, pred in zip(y, preds) if (yi > 0) == (pred.data > 0)) / N
    
    losses.append(total_loss.data)
    accuracies.append(acc)
    
    # Backward pass
    # Resetear gradientes
    for p in model.parameters():
        p.grad = 0.0
    total_loss.backward()
    
    # SGD update
    for p in model.parameters():
        p.data -= lr * p.grad
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {total_loss.data:.4f} | Accuracy: {acc:.2%}')

print(f'\nEntrenamiento completo!')
print(f'Loss final: {losses[-1]:.4f}')
print(f'Accuracy final: {accuracies[-1]:.2%}')

In [ ]:
# Visualizar entrenamiento y frontera de decision
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(losses, color='#e63946', linewidth=2)
axes[0].set_title('Perdida durante Entrenamiento', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(accuracies, color='#2d6a4f', linewidth=2)
axes[1].set_title('Accuracy durante Entrenamiento', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3)

# Decision boundary
h = 0.1
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = np.array([model([Value(xi), Value(yi)]).data for xi, yi in zip(xx.ravel(), yy.ravel())])
Z = Z.reshape(xx.shape)

axes[2].contourf(xx, yy, Z, levels=[-1e9, 0, 1e9], colors=['#e6394633', '#457b9d33'])
axes[2].contour(xx, yy, Z, levels=[0], colors=['black'], linewidths=2)
axes[2].scatter(X[y == -1, 0], X[y == -1, 1], c='#e63946', s=30, edgecolors='white', linewidth=0.5)
axes[2].scatter(X[y == 1, 0], X[y == 1, 1], c='#457b9d', s=30, edgecolors='white', linewidth=0.5)
axes[2].set_title('Frontera de Decision Aprendida', fontsize=13, fontweight='bold')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.savefig('entrenamiento_autograd.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Parte 5: Como lo Hace Tinygrad?

Nuestro motor autograd funciona, pero opera con escalares. Tinygrad lleva esto al siguiente nivel:

### Comparacion: Nuestro Autograd vs Tinygrad

| Aspecto | Nuestro Autograd | Tinygrad |
|---------|-----------------|----------|
| **Datos** | Escalares | Tensores N-dimensionales |
| **Ejecucion** | Eager (inmediata) | Lazy (diferida) |
| **Backend** | Solo CPU/Python | GPU, CPU, Metal, CUDA |
| **Fusion** | No | Fusion de kernels |
| **Lineas de codigo** | ~100 | ~10,000-19,000 |
| **Modelos reales** | No | LLaMA-65B, Stable Diffusion |

### Arquitectura de Tinygrad

```
Tensor API (usuario)
     |
     v
UOp Graph (grafo de operaciones universales)
     |
     v
Scheduler (fusion de kernels)
     |
     v
Codegen (generacion de codigo CUDA/Metal/C)
     |
     v
Hardware (GPU/CPU/TPU)
```

In [ ]:
# Simulacion del concepto de Lazy Evaluation de Tinygrad

class LazyTensor:
    """Simula la evaluacion lazy de tinygrad.
    Las operaciones NO se ejecutan hasta llamar .realize()"""
    
    def __init__(self, data=None, op=None, sources=None, label=''):
        self._data = data  # None si es lazy
        self._op = op
        self._sources = sources or []
        self.label = label
        self._realized = data is not None
    
    @property
    def data(self):
        if not self._realized:
            self.realize()
        return self._data
    
    def __add__(self, other):
        print(f'  [LAZY] Registrando operacion: {self.label} + {other.label}')
        return LazyTensor(op='add', sources=[self, other], label=f'({self.label}+{other.label})')
    
    def __mul__(self, other):
        print(f'  [LAZY] Registrando operacion: {self.label} * {other.label}')
        return LazyTensor(op='mul', sources=[self, other], label=f'({self.label}*{other.label})')
    
    def sum(self):
        print(f'  [LAZY] Registrando operacion: sum({self.label})')
        return LazyTensor(op='sum', sources=[self], label=f'sum({self.label})')
    
    def realize(self):
        """Ejecuta todas las operaciones pendientes (como tinygrad)."""
        if self._realized:
            return self._data
        
        # Realizar fuentes primero
        source_data = [s.realize() for s in self._sources]
        
        if self._op == 'add':
            self._data = source_data[0] + source_data[1]
        elif self._op == 'mul':
            self._data = source_data[0] * source_data[1]
        elif self._op == 'sum':
            self._data = np.sum(source_data[0])
        
        self._realized = True
        print(f'  [REALIZE] Ejecutando: {self.label} = {self._data}')
        return self._data


print('=== DEMOSTRACION: Eager vs Lazy ===' )
print()
print('--- EAGER (como PyTorch/nuestro autograd) ---')
a_np = np.array([1.0, 2.0, 3.0])
b_np = np.array([4.0, 5.0, 6.0])
c_np = a_np + b_np
print(f'  a + b = {c_np} (ejecutado inmediatamente)')
d_np = c_np * a_np
print(f'  c * a = {d_np} (ejecutado inmediatamente)')
e_np = np.sum(d_np)
print(f'  sum = {e_np} (ejecutado inmediatamente)')
print(f'  Total: 3 operaciones separadas en memoria\n')

print('--- LAZY (como Tinygrad) ---')
a = LazyTensor(np.array([1.0, 2.0, 3.0]), label='a')
b = LazyTensor(np.array([4.0, 5.0, 6.0]), label='b')
c = a + b
d = c * a
e = d.sum()
print(f'  (Nada ejecutado aun - solo se registro el grafo)\n')
print('  Llamando .realize():')
result = e.realize()
print(f'\n  Ventaja: Tinygrad puede FUSIONAR estas 3 ops en 1 kernel GPU!')

---
## Parte 6: Operaciones Fundamentales (Ops) de Tinygrad

Tinygrad reduce TODAS las operaciones de deep learning a un conjunto minimo de operaciones primitivas. Esta es una de sus innovaciones clave.

### Las categorias de operaciones:

| Categoria | Operaciones | Ejemplo |
|-----------|------------|---------|
| **Unarias** | neg, exp, log, sin, sqrt | `x.exp()` |
| **Binarias** | add, mul, max, cmplt | `a + b` |
| **Reduce** | sum, max | `x.sum(axis=0)` |
| **Movement** | reshape, permute, expand, pad | `x.reshape(2,3)` |
| **Load/Store** | load, store, const | I/O con memoria |

In [ ]:
# Implementacion de las operaciones primitivas con autograd

class TensorValue:
    """Tensor con autograd usando operaciones primitivas como tinygrad."""
    
    def __init__(self, data, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float32)
        self.grad = np.zeros_like(self.data, dtype=np.float32)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    def __repr__(self):
        return f"TensorValue(shape={self.data.shape}, op={self._op})"
    
    def _reduce_grad(self, grad, target_shape):
        """Reduce gradiente sumando ejes que fueron broadcast."""
        while len(grad.shape) > len(target_shape):
            grad = grad.sum(axis=0)
        for i, (gs, ts) in enumerate(zip(grad.shape, target_shape)):
            if ts == 1 and gs != 1:
                grad = grad.sum(axis=i, keepdims=True)
        return grad

    # --- Operaciones Unarias ---
    def neg(self):
        out = TensorValue(-self.data, (self,), 'neg')
        def _backward():
            self.grad += -out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        e = np.exp(self.data)
        out = TensorValue(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out
    
    def log(self):
        out = TensorValue(np.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = TensorValue(np.maximum(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (self.data > 0).astype(np.float32) * out.grad
        out._backward = _backward
        return out
    
    # --- Operaciones Binarias ---
    def __add__(self, other):
        other = other if isinstance(other, TensorValue) else TensorValue(other)
        out = TensorValue(self.data + other.data, (self, other), 'add')
        def _backward():
            self.grad += self._reduce_grad(out.grad, self.data.shape)
            other.grad += other._reduce_grad(out.grad, other.data.shape)
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, TensorValue) else TensorValue(other)
        out = TensorValue(self.data * other.data, (self, other), 'mul')
        def _backward():
            self.grad += self._reduce_grad(other.data * out.grad, self.data.shape)
            other.grad += other._reduce_grad(self.data * out.grad, other.data.shape)
        out._backward = _backward
        return out
    
    # --- Operaciones Reduce ---
    def sum(self, axis=None):
        out = TensorValue(np.sum(self.data, axis=axis), (self,), 'sum')
        def _backward():
            if axis is None:
                self.grad += np.ones_like(self.data) * out.grad
            else:
                self.grad += np.expand_dims(out.grad, axis=axis) * np.ones_like(self.data)
        out._backward = _backward
        return out
    
    def mean(self):
        n = self.data.size
        out = TensorValue(np.mean(self.data), (self,), 'mean')
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad / n
        out._backward = _backward
        return out
    
    # --- Matmul ---
    def matmul(self, other):
        out = TensorValue(self.data @ other.data, (self, other), 'matmul')
        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = np.ones_like(self.data)
        for node in reversed(topo):
            node._backward()

print('TensorValue definido con operaciones primitivas al estilo tinygrad')

In [ ]:
# Demo: Regresion lineal con TensorValue
np.random.seed(42)

# Datos sinteticos: y = 3x1 + 2x2 + 1 + ruido
X_data = np.random.randn(50, 2).astype(np.float32)
y_data = (3 * X_data[:, 0] + 2 * X_data[:, 1] + 1 + np.random.randn(50) * 0.1).reshape(-1, 1).astype(np.float32)

# Parametros a aprender
W = TensorValue(np.random.randn(2, 1).astype(np.float32) * 0.1)
b = TensorValue(np.zeros((1, 1), dtype=np.float32))

lr = 0.01
losses_lr = []

for epoch in range(200):
    X_t = TensorValue(X_data)
    y_t = TensorValue(y_data)
    
    # Forward: y_pred = X @ W + b
    pred = X_t.matmul(W) + TensorValue(np.ones((50, 1), dtype=np.float32)) * b
    
    # MSE Loss
    diff = pred + y_t.neg()
    loss = (diff * diff).mean()
    
    losses_lr.append(loss.data.item())
    
    # Backward
    W.grad = np.zeros_like(W.data)
    b.grad = np.zeros_like(b.data)
    loss.backward()
    
    # SGD
    W.data -= lr * W.grad
    b.data -= lr * b.grad
    
    if (epoch + 1) % 40 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss.data.item():.6f} | W: [{W.data[0,0]:.3f}, {W.data[1,0]:.3f}] | b: {b.data[0,0]:.3f}')

print(f'\nParametros aprendidos: W = [{W.data[0,0]:.3f}, {W.data[1,0]:.3f}], b = {b.data[0,0]:.3f}')
print(f'Parametros reales:     W = [3.000, 2.000], b = 1.000')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_lr, color='#2d6a4f', linewidth=2)
ax.set_title('Regresion Lineal con TensorValue (Autograd propio)', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Parte 7: Visualizacion del Flujo de Gradientes

Veamos como fluyen los gradientes a traves de una red simple, paso a paso.

In [ ]:
# Visualizacion paso a paso del backpropagation
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Red simple: 2 -> 2 -> 1
# Forward: h1 = relu(w1*x1 + w2*x2), h2 = relu(w3*x1 + w4*x2), y = w5*h1 + w6*h2

x1_v = 1.0; x2_v = 0.5
w1_v = 0.5; w2_v = -0.3; w3_v = 0.8; w4_v = 0.2; w5_v = 0.6; w6_v = -0.4

# Forward pass con registro
z1 = w1_v * x1_v + w2_v * x2_v  # = 0.35
z2 = w3_v * x1_v + w4_v * x2_v  # = 0.9
h1 = max(0, z1)  # relu = 0.35
h2 = max(0, z2)  # relu = 0.9
y_pred = w5_v * h1 + w6_v * h2  # = 0.21 - 0.36 = -0.15
y_true = 1.0
loss = (y_pred - y_true) ** 2  # MSE

# Backward pass manual
dL_dy = 2 * (y_pred - y_true)  # = 2*(-1.15) = -2.3
dL_dw5 = dL_dy * h1
dL_dw6 = dL_dy * h2
dL_dh1 = dL_dy * w5_v
dL_dh2 = dL_dy * w6_v
dL_dz1 = dL_dh1 * (1 if z1 > 0 else 0)  # relu grad
dL_dz2 = dL_dh2 * (1 if z2 > 0 else 0)
dL_dw1 = dL_dz1 * x1_v
dL_dw2 = dL_dz1 * x2_v
dL_dw3 = dL_dz2 * x1_v
dL_dw4 = dL_dz2 * x2_v

def draw_network(ax, step, values, grads, title, highlight_layer=-1):
    """Dibuja la red con valores y gradientes."""
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, 2.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Posiciones de nodos
    positions = {
        'x1': (0, 2), 'x2': (0, 0.5),
        'h1': (1.5, 2), 'h2': (1.5, 0.5),
        'y': (3, 1.25)
    }
    
    # Colores por capa
    layers = {0: ['x1', 'x2'], 1: ['h1', 'h2'], 2: ['y']}
    colors_base = {0: '#0e4d92', 1: '#2d6a4f', 2: '#e63946'}
    
    for layer_idx, nodes in layers.items():
        for node in nodes:
            px, py = positions[node]
            color = '#ffaa00' if layer_idx == highlight_layer else colors_base[layer_idx]
            circle = plt.Circle((px, py), 0.3, color=color, alpha=0.8)
            ax.add_patch(circle)
            v = values.get(node, '')
            g = grads.get(node, '')
            ax.text(px, py + 0.05, f'{node}', ha='center', va='bottom', fontsize=9, 
                   fontweight='bold', color='white')
            if v != '':
                ax.text(px, py - 0.1, f'{v:.2f}', ha='center', va='top', fontsize=8, color='white')
            if g != '':
                ax.text(px, py - 0.45, f'g={g:.2f}', ha='center', va='top', fontsize=7, 
                       color='#ffcc44', fontstyle='italic')
    
    # Conexiones
    connections = [('x1', 'h1'), ('x1', 'h2'), ('x2', 'h1'), ('x2', 'h2'), ('h1', 'y'), ('h2', 'y')]
    for src, dst in connections:
        sx, sy = positions[src]
        dx, dy = positions[dst]
        ax.annotate('', xy=(dx - 0.3, dy), xytext=(sx + 0.3, sy),
                   arrowprops=dict(arrowstyle='->', color='#888', lw=1))

# Paso 1: Red inicial
draw_network(axes[0, 0], 1, 
            {'x1': x1_v, 'x2': x2_v}, {},
            'Paso 1: Entradas', 0)

# Paso 2: Forward - capa oculta
draw_network(axes[0, 1], 2,
            {'x1': x1_v, 'x2': x2_v, 'h1': h1, 'h2': h2}, {},
            'Paso 2: Forward (oculta)', 1)

# Paso 3: Forward - salida
draw_network(axes[0, 2], 3,
            {'x1': x1_v, 'x2': x2_v, 'h1': h1, 'h2': h2, 'y': y_pred}, {},
            f'Paso 3: Forward (salida), Loss={loss:.3f}', 2)

# Paso 4: Backward - gradiente de salida
draw_network(axes[1, 0], 4,
            {'x1': x1_v, 'x2': x2_v, 'h1': h1, 'h2': h2, 'y': y_pred},
            {'y': dL_dy},
            'Paso 4: Backward (salida)', 2)

# Paso 5: Backward - gradientes capa oculta
draw_network(axes[1, 1], 5,
            {'x1': x1_v, 'x2': x2_v, 'h1': h1, 'h2': h2, 'y': y_pred},
            {'y': dL_dy, 'h1': dL_dh1, 'h2': dL_dh2},
            'Paso 5: Backward (oculta)', 1)

# Paso 6: Backward - todas
draw_network(axes[1, 2], 6,
            {'x1': x1_v, 'x2': x2_v, 'h1': h1, 'h2': h2, 'y': y_pred},
            {'y': dL_dy, 'h1': dL_dh1, 'h2': dL_dh2, 'x1': dL_dw1 + dL_dw3, 'x2': dL_dw2 + dL_dw4},
            'Paso 6: Gradientes completos', -1)

plt.suptitle('Visualizacion Paso a Paso: Forward + Backward Pass',
            fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('flujo_gradientes.png', dpi=150, bbox_inches='tight')
plt.show()

print('Gradientes calculados manualmente:')
print(f'  dL/dw1 = {dL_dw1:.4f}, dL/dw2 = {dL_dw2:.4f}')
print(f'  dL/dw3 = {dL_dw3:.4f}, dL/dw4 = {dL_dw4:.4f}')
print(f'  dL/dw5 = {dL_dw5:.4f}, dL/dw6 = {dL_dw6:.4f}')

---
## Parte 8: Fusion de Kernels - La Ventaja de Tinygrad

Una de las innovaciones principales de tinygrad es la **fusion de kernels**: combinar multiples operaciones en un solo kernel de GPU para minimizar accesos a memoria.

In [ ]:
# Simulacion visual de fusion de kernels
import time

N_sim = 1_000_000
a_sim = np.random.randn(N_sim).astype(np.float32)
b_sim = np.random.randn(N_sim).astype(np.float32)

# Sin fusion: operaciones separadas (como PyTorch eager)
t0 = time.perf_counter()
for _ in range(100):
    c_sim = a_sim + b_sim       # Op 1: lee a,b de memoria, escribe c
    d_sim = c_sim * 2.0         # Op 2: lee c de memoria, escribe d
    e_sim = np.maximum(d_sim, 0) # Op 3: lee d de memoria, escribe e
    f_sim = np.sum(e_sim)       # Op 4: lee e de memoria, escribe f
t_separate = time.perf_counter() - t0

# Con fusion: una sola pasada (como tinygrad compilado)
t0 = time.perf_counter()
for _ in range(100):
    # Operacion fusionada equivalente
    f_fused = np.sum(np.maximum((a_sim + b_sim) * 2.0, 0))
t_fused = time.perf_counter() - t0

speedup = t_separate / t_fused

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Diagrama sin fusion
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title(f'Sin Fusion (Eager): {t_separate:.3f}s', fontsize=13, fontweight='bold', color='#e63946')
ax.axis('off')

steps = [
    ('Memoria', 'a, b', 8.5, '#4a4e69'),
    ('Kernel 1', 'c = a + b', 6.5, '#e63946'),
    ('Memoria', 'c', 5.5, '#4a4e69'),
    ('Kernel 2', 'd = c * 2', 4.0, '#e63946'),
    ('Memoria', 'd', 3.0, '#4a4e69'),
    ('Kernel 3', 'e = relu(d)', 1.5, '#e63946'),
]
for label, desc, ypos, color in steps:
    rect = plt.Rectangle((1, ypos - 0.4), 8, 0.8, color=color, alpha=0.7, ec='white')
    ax.add_patch(rect)
    ax.text(5, ypos, f'{label}: {desc}', ha='center', va='center', 
           fontsize=11, color='white', fontweight='bold')

for i in range(len(steps) - 1):
    y1 = steps[i][2] - 0.4
    y2 = steps[i+1][2] + 0.4
    ax.annotate('', xy=(5, y2), xytext=(5, y1),
               arrowprops=dict(arrowstyle='->', color='#ffaa00', lw=2))

ax.text(5, 0.5, '6 accesos a memoria global', ha='center', fontsize=10, 
       color='#ffaa00', fontstyle='italic')

# Diagrama con fusion
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title(f'Con Fusion (Tinygrad): {t_fused:.3f}s ({speedup:.1f}x)', 
            fontsize=13, fontweight='bold', color='#2d6a4f')
ax.axis('off')

rect = plt.Rectangle((1, 7.5), 8, 0.8, color='#4a4e69', alpha=0.7, ec='white')
ax.add_patch(rect)
ax.text(5, 7.9, 'Memoria: a, b', ha='center', va='center', fontsize=11, 
       color='white', fontweight='bold')

rect2 = plt.Rectangle((1, 3), 8, 3.5, color='#2d6a4f', alpha=0.7, ec='white', linewidth=2)
ax.add_patch(rect2)
ax.text(5, 5.5, 'KERNEL FUSIONADO', ha='center', va='center', fontsize=13, 
       color='white', fontweight='bold')
ax.text(5, 4.5, 'f = sum(relu((a+b)*2))', ha='center', va='center', fontsize=11, 
       color='#aaffaa')
ax.text(5, 3.5, '3 ops en 1 kernel', ha='center', va='center', fontsize=10, 
       color='#cccccc')

ax.annotate('', xy=(5, 6.5), xytext=(5, 7.1),
           arrowprops=dict(arrowstyle='->', color='#ffaa00', lw=2))

rect3 = plt.Rectangle((1, 1.5), 8, 0.8, color='#4a4e69', alpha=0.7, ec='white')
ax.add_patch(rect3)
ax.text(5, 1.9, 'Memoria: f (escalar)', ha='center', va='center', fontsize=11, 
       color='white', fontweight='bold')

ax.annotate('', xy=(5, 2.3), xytext=(5, 3.0),
           arrowprops=dict(arrowstyle='->', color='#ffaa00', lw=2))

ax.text(5, 0.7, '2 accesos a memoria global', ha='center', fontsize=10, 
       color='#2d6a4f', fontstyle='italic', fontweight='bold')

plt.suptitle('Fusion de Kernels: Menos Accesos a Memoria = Mas Velocidad',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fusion_kernels.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Parte 9: Comparativa Final - Ecosistema de Frameworks

### Ruta de aprendizaje recomendada:
1. **Micrograd** (~100 lineas): Backpropagation escalar
2. **Nuestro autograd** (este notebook): Backprop escalar + tensores
3. **Tinygrad** (~19K lineas): Framework completo con GPU, fusion de kernels
4. **PyTorch** (millones de lineas): Framework de produccion

In [ ]:
# Tabla comparativa visual
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

frameworks = [
    ['', 'Micrograd', 'Nuestro Autograd', 'Tinygrad', 'PyTorch'],
    ['Lineas de codigo', '~100', '~200', '~19,000', 'Millones'],
    ['Tipo de datos', 'Escalar', 'Escalar + Tensor', 'Tensor N-D', 'Tensor N-D'],
    ['Ejecucion', 'Eager', 'Eager', 'Lazy', 'Eager + JIT'],
    ['GPU', 'No', 'No', 'CUDA, Metal, OpenCL', 'CUDA'],
    ['Fusion de kernels', 'No', 'No', 'Si (automatica)', 'Si (torch.compile)'],
    ['Modelos reales', 'No', 'Toy models', 'LLaMA, StableDiff', 'Todos'],
    ['Autograd', 'Reverse mode', 'Reverse mode', 'IR-based autodiff', 'Dynamic graph'],
    ['Valor educativo', 'Fundamentos', 'Intermedio', 'Avanzado', 'Produccion'],
]

colors_header = ['#1a1a2e', '#e63946', '#457b9d', '#2d6a4f', '#ff8c00']
colors_row = ['#1a1a2e22', '#f8f8f8']

table = ax.table(cellText=frameworks, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)

# Estilizar header
for j in range(5):
    cell = table[0, j]
    cell.set_facecolor(colors_header[j])
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)

# Estilizar filas
for i in range(1, len(frameworks)):
    for j in range(5):
        cell = table[i, j]
        if j == 0:
            cell.set_text_props(fontweight='bold')
            cell.set_facecolor('#e8e8e8')
        else:
            cell.set_facecolor(colors_row[i % 2])

ax.set_title('Comparativa de Frameworks de Deep Learning',
            fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('comparativa_frameworks.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Parte 10: Valor de Negocio de Tinygrad

### Por que importa para la industria?

1. **Reduccion de dependencias**: Tinygrad no depende de LLVM/CUDA toolkit pesados
2. **Hardware agnostico**: Soporte nativo para GPU AMD, Apple Silicon, Intel
3. **Inference eficiente**: Fusion de kernels reduce costos de compute en produccion
4. **Modelo de negocio (tiny corp)**:
   - Desarrollando chips AI propios
   - Cloud compute basado en tinygrad
   - Alternativa open-source a CUDA lock-in de NVIDIA

### Casos de uso empresarial:
- **Edge deployment**: Framework liviano ideal para dispositivos embebidos
- **Multi-GPU/Multi-vendor**: Mismo codigo para NVIDIA, AMD, Apple
- **Costos de compute**: Fusion automatica reduce uso de memoria y tiempo
- **Auditabilidad**: Codigo legible facilita compliance y debugging

In [ ]:
print('=' * 70)
print('     CONCLUSIONES - Tinygrad: Autograd y Grafos de Computacion')
print('=' * 70)
print()
print('1. AUTOGRAD no es magia: es la regla de la cadena aplicada')
print('   sistematicamente sobre un grafo de computacion (DAG).')
print()
print('2. GRAFOS DE COMPUTACION son la columna vertebral de todo')
print('   framework de deep learning. Permiten:')
print('   - Backpropagation automatica')
print('   - Optimizacion (fusion de kernels)')
print('   - Ejecucion en multiples backends')
print()
print('3. TINYGRAD demuestra que un framework completo puede existir')
print('   en ~19K lineas, ejecutando modelos como LLaMA-65B.')
print()
print('4. LAZY EVALUATION permite optimizaciones imposibles en modo')
print('   eager: el grafo completo se conoce antes de ejecutar.')
print()
print('5. VALOR DE NEGOCIO: romper el vendor lock-in de NVIDIA CUDA,')
print('   reducir costos de inference, y deployar en cualquier hardware.')
print()
print('=' * 70)
print('Repositorio: https://github.com/tinygrad/tinygrad')
print('=' * 70)